## 1. Setup and Imports

In [1]:
import pandas as pd
import numpy as np
from typing import List, Tuple, Dict
import ezdxf
import os

print('All imports successful!')

All imports successful!


## 2. Load Data

In [2]:
objects_df = pd.read_csv('../data/test_objects_250.csv')

print(f'Loaded {len(objects_df)} object types')
print(f'\nCategories:')
print(objects_df['category'].value_counts())
print(f'\nStackable objects:')
print(objects_df[objects_df['stackable'] == True][['name', 'stackable', 'max_stack_height']])

Loaded 250 object types

Categories:
category
Concrete    80
Steel       60
Wood        50
Plastic     40
Glass       20
Name: count, dtype: int64

Stackable objects:
                   name  stackable  max_stack_height
0           Steel Plate       True                10
1            Steel Pipe       True                 8
2           Steel Plate       True                10
3    Steel Rebar Bundle       True                 6
4            Steel Beam       True                 5
..                  ...        ...               ...
225       Plastic Sheet       True                25
226         Plastic Bin       True                30
227         Plastic Bin       True                30
228   Plastic Container       True                20
229       Plastic Sheet       True                25

[230 rows x 3 columns]


### 2.5 Segregate by Category

In [3]:
# SEGREGATION RULES - Flexible System with User Input
print("=== SETTING UP SEGREGATION RULES ===\n")

# Get categories from data
all_categories = sorted(objects_df['category'].unique().tolist())
print(f"Available categories: {all_categories}\n")

print("SEGREGATION RULE OPTIONS:")
print("1. No segregation (all categories can mix)")
print("2. Define incompatible pairs (e.g., Glass ❌ Concrete)")
print("3. Define compatible pairs (e.g., Steel ✓ Concrete only)")
print("4. Use preset rules\n")

choice = input("Select option (1-4): ").strip()

if choice == '1':
    # All categories compatible with each other
    compatibility_matrix = {}
    for cat in all_categories:
        compatibility_matrix[cat] = {other: True for other in all_categories}
    print("✓ No segregation - all categories can mix\n")

elif choice == '2':
    # Define incompatible pairs
    print("\nDefine incompatible pairs (e.g., 'Glass,Concrete' or 'Steel,Wood')")
    print("Enter pairs separated by comma. Type 'done' when finished.\n")
    
    incompatible_pairs = []
    while True:
        pair_input = input("Enter pair (or 'done'): ").strip()
        if pair_input.lower() == 'done':
            break
        
        try:
            cat1, cat2 = [p.strip() for p in pair_input.split(',')]
            if cat1 in all_categories and cat2 in all_categories:
                incompatible_pairs.append((cat1, cat2))
                print(f"  ✓ {cat1} ❌ {cat2}")
            else:
                print(f"  ❌ Invalid category. Use: {all_categories}")
        except:
            print("  ❌ Invalid format. Use: 'Category1,Category2'")
    
    # Build compatibility matrix from incompatible pairs
    compatibility_matrix = {}
    for cat in all_categories:
        compatibility_matrix[cat] = {}
        for other_cat in all_categories:
            is_incompatible = (cat, other_cat) in incompatible_pairs or (other_cat, cat) in incompatible_pairs
            compatibility_matrix[cat][other_cat] = not is_incompatible
    
    print("\n✓ Incompatible pairs configured")

elif choice == '3':
    # Define compatible pairs
    print("\nFor each category, define what it can be with.")
    print(f"(Separate multiple with commas, e.g., 'Steel,Concrete')\n")
    
    compatible_with = {}
    for cat in all_categories:
        while True:
            compatible_input = input(f"{cat} can be with: ").strip()
            
            if not compatible_input:
                print("  (At least include the category itself)")
                continue
            
            allowed = [c.strip() for c in compatible_input.split(',')]
            
            # Validate
            invalid = [c for c in allowed if c not in all_categories]
            if invalid:
                print(f"  ❌ Invalid categories: {invalid}. Use: {all_categories}")
                continue
            
            compatible_with[cat] = allowed
            print(f"  ✓ {cat}: {allowed}")
            break
    
    # Build compatibility matrix
    compatibility_matrix = {}
    for cat in all_categories:
        compatibility_matrix[cat] = {}
        for other_cat in all_categories:
            compatibility_matrix[cat][other_cat] = other_cat in compatible_with.get(cat, [])
    
    print("\n✓ Compatible pairs configured")

elif choice == '4':
    # Use preset rules
    print("\nPRESET RULES:")
    print("A. Glass isolated, Steel/Concrete together, Wood/Plastic together")
    print("B. All separate (only same category together)")
    print("C. All compatible\n")
    
    preset = input("Select preset (A, B, or C): ").strip().upper()
    
    compatible_with = {}
    
    if preset == 'A':
        compatible_with = {
            'Glass': ['Glass'],
            'Concrete': ['Concrete', 'Steel'],
            'Steel': ['Concrete', 'Steel'],
            'Wood': ['Wood', 'Plastic'],
            'Plastic': ['Wood', 'Plastic'],
        }
        print("✓ Using Preset A (Glass isolated, Steel/Concrete, Wood/Plastic)")
    
    elif preset == 'B':
        compatible_with = {cat: [cat] for cat in all_categories}
        print("✓ Using Preset B (All categories separate)")
    
    elif preset == 'C':
        compatible_with = {cat: all_categories for cat in all_categories}
        print("✓ Using Preset C (All compatible)")
    
    else:
        print("Invalid preset, using C (all compatible)")
        compatible_with = {cat: all_categories for cat in all_categories}
    
    # Build compatibility matrix
    compatibility_matrix = {}
    for cat in all_categories:
        compatibility_matrix[cat] = {}
        for other_cat in all_categories:
            compatibility_matrix[cat][other_cat] = other_cat in compatible_with.get(cat, [])

else:
    print("Invalid choice, using no segregation")
    compatibility_matrix = {}
    for cat in all_categories:
        compatibility_matrix[cat] = {other: True for other in all_categories}

# Display final matrix with checkmarks and X's
print("\n" + "="*60)
print("FINAL COMPATIBILITY MATRIX:")
print("="*60)

# Create visual matrix
all_cats = sorted(all_categories)
col_width = max(len(cat) for cat in all_cats) + 2

# Header row
header = "".ljust(col_width)
for cat in all_cats:
    header += cat.ljust(col_width)
print(header)

# Separator
print("-" * (col_width * (len(all_cats) + 1)))

# Matrix rows
for cat1 in all_cats:
    row = cat1.ljust(col_width)
    for cat2 in all_cats:
        if compatibility_matrix[cat1][cat2]:
            row += "✓".ljust(col_width)
        else:
            row += "❌".ljust(col_width)
    print(row)

print("\n" + "="*60)
for category in all_cats:
    allowed = [c for c in all_cats if compatibility_matrix[category][c]]
    print(f"{category}: {allowed}")

print(f"\n✓ Segregation rules ready")

=== SETTING UP SEGREGATION RULES ===

Available categories: ['Concrete', 'Glass', 'Plastic', 'Steel', 'Wood']

SEGREGATION RULE OPTIONS:
1. No segregation (all categories can mix)
2. Define incompatible pairs (e.g., Glass ❌ Concrete)
3. Define compatible pairs (e.g., Steel ✓ Concrete only)
4. Use preset rules

✓ No segregation - all categories can mix


FINAL COMPATIBILITY MATRIX:
          Concrete  Glass     Plastic   Steel     Wood      
------------------------------------------------------------
Concrete  ✓         ✓         ✓         ✓         ✓         
Glass     ✓         ✓         ✓         ✓         ✓         
Plastic   ✓         ✓         ✓         ✓         ✓         
Steel     ✓         ✓         ✓         ✓         ✓         
Wood      ✓         ✓         ✓         ✓         ✓         

Concrete: ['Concrete', 'Glass', 'Plastic', 'Steel', 'Wood']
Glass: ['Concrete', 'Glass', 'Plastic', 'Steel', 'Wood']
Plastic: ['Concrete', 'Glass', 'Plastic', 'Steel', 'Wood']
Steel: ['Con

## 3. Define 2D Bin Packing Classes

In [4]:
class Rectangle:
    def __init__(self, width, length, obj_id, obj_name, category, weight, stackable=False, max_stack=1):
        self.width = width
        self.length = length
        self.obj_id = obj_id
        self.obj_name = obj_name
        self.category = category
        self.weight = weight
        self.stackable = stackable
        self.max_stack = max_stack
        self.x = 0
        self.y = 0
        self.rotated = False
        self.stack_height = 1
    
    def get_width(self):
        return self.length if self.rotated else self.width
    
    def get_length(self):
        return self.width if self.rotated else self.length
    
    def area(self):
        return self.width * self.length
    
    def __repr__(self):
        return f"Rectangle({self.obj_name}, {self.width}x{self.length})"

print('✓ Rectangle class defined')

✓ Rectangle class defined


In [5]:
class Bin:
    def __init__(self, width, length, bin_id, min_spacing=0, compatibility_matrix=None):
        self.width = width
        self.length = length
        self.bin_id = bin_id
        self.rectangles = []
        self.min_spacing = min_spacing
        self.compatibility_matrix = compatibility_matrix or {}
    
    def can_fit(self, rect):
        for existing in self.rectangles:
            if self._overlaps(rect, existing):
                return False
        
        # Check compatibility
        if self.compatibility_matrix:
            for existing in self.rectangles:
                if not self.compatibility_matrix.get(rect.category, {}).get(existing.category, True):
                    return False
        
        # Check if rect fits in bin with spacing
        if rect.x + rect.get_width() + self.min_spacing > self.width:
            return False
        if rect.y + rect.get_length() + self.min_spacing > self.length:
            return False
        
        return True
    
    def can_accept_category(self, category):
        """Check if this bin can accept a new category."""
        if not self.rectangles:
            return True
        
        if not self.compatibility_matrix:
            return True
        
        # Check compatibility with all existing categories in bin
        existing_categories = set(rect.category for rect in self.rectangles)
        
        for existing_cat in existing_categories:
            if not self.compatibility_matrix.get(category, {}).get(existing_cat, True):
                return False
        
        return True
    
    def _overlaps(self, rect1, rect2):
        return not (rect1.x + rect1.get_width() + self.min_spacing <= rect2.x or 
                   rect1.x >= rect2.x + rect2.get_width() + self.min_spacing or
                   rect1.y + rect1.get_length() + self.min_spacing <= rect2.y or 
                   rect1.y >= rect2.y + rect2.get_length() + self.min_spacing)

print('✓ Bin class updated with segregation support')

✓ Bin class updated with segregation support


## 4. Implement Packing Algorithm

In [6]:
from cmath import rect
from copy import deepcopy

def pack_objects(objects_df, bin_width=100, bin_length=100, min_spacing=0, compatibility_matrix=None):
    """
    Pack objects using 2D bin packing with segregation rules.
    Process ALL objects together instead of by category.
    """
    print(f"\nProcessing ALL objects with segregation rules...")
    
    # Create rectangles for ALL objects (not by category)
    rects = []
    for _, row in objects_df.iterrows():
        rect = Rectangle(
            width=float(row['width']),
            length=float(row['length']),
            obj_id=row['object_id'],
            obj_name=row['name'],
            category=row['category'],
            weight=float(row['weight']),
            stackable=row['stackable'] in [True, 'True', 'TRUE'],
            max_stack=int(row['max_stack_height'])
        )
        rects.append(rect)
    
    # Sort by area (largest first)
    rects.sort(key=lambda r: r.area(), reverse=True)
    
    # Pack into bins
    bins = []
    
    for rect in rects:
        placed = False
        
       # Try to stack on existing object of same type
        if rect.stackable:
            for bin_idx, bin_obj in enumerate(bins):
                if placed:
                    break
                for existing in bin_obj.rectangles:
                    # Stack if same name, dimensions, and stackable
                    if (existing.obj_name == rect.obj_name and 
                        existing.width == rect.width and
                        existing.length == rect.length and
                        existing.stack_height < existing.max_stack):
                        # Check compatibility
                        if compatibility_matrix:
                            if not compatibility_matrix.get(rect.category, {}).get(existing.category, True):
                                continue
                            
                        # Stack on top - SAME ORIENTATION as base
                        rect.x = existing.x
                        rect.y = existing.y
                        rect.rotated = existing.rotated  # Match base orientation!
                        rect.stack_height = existing.stack_height + 1
                        existing.stack_height = rect.stack_height
                        bin_obj.rectangles.append(rect)
                        placed = True
                        print(f"  Stacked: {rect.obj_name} ({rect.category}) - Stack height: {rect.stack_height}")
                        break
                    
        # If not stacked, try to fit in existing bin
        if not placed:
            for bin_idx, bin_obj in enumerate(bins):
                if placed:
                    break
                    
                # Check if this category can go in this bin
                if compatibility_matrix:
                    if not bin_obj.can_accept_category(rect.category):
                        continue
                
                # Try different positions
                positions_to_try = [(0, 0)]
                for existing in bin_obj.rectangles:
                    positions_to_try.append((existing.x + existing.get_width() + min_spacing, existing.y))
                    positions_to_try.append((existing.x, existing.y + existing.get_length() + min_spacing))
                
                for x, y in positions_to_try:
                    if placed:
                        break
                    
                    # Try normal orientation
                    rect.x = x
                    rect.y = y
                    rect.rotated = False
                    
                    # Check if fits with spacing
                    fits = True
                    if rect.x + rect.get_width() + min_spacing > bin_obj.width:
                        fits = False
                    if rect.y + rect.get_length() + min_spacing > bin_obj.length:
                        fits = False
                    
                    # Check overlap
                    if fits:
                        for existing in bin_obj.rectangles:
                            if bin_obj._overlaps(rect, existing):
                                fits = False
                                break
                    
                    if fits:
                        bin_obj.rectangles.append(rect)
                        placed = True
                        break
                    
                    # Try rotated orientation
                    rect.rotated = True
                    fits = True
                    if rect.x + rect.get_width() + min_spacing > bin_obj.width:
                        fits = False
                    if rect.y + rect.get_length() + min_spacing > bin_obj.length:
                        fits = False
                    
                    if fits:
                        for existing in bin_obj.rectangles:
                            if bin_obj._overlaps(rect, existing):
                                fits = False
                                break
                    
                    if fits:
                        bin_obj.rectangles.append(rect)
                        placed = True
                        break
                    
                    rect.rotated = False
        
        # Create new bin if needed
        if not placed:
            new_bin = Bin(bin_width, bin_length, len(bins), min_spacing=min_spacing, 
                         compatibility_matrix=compatibility_matrix)
            rect.x = 0
            rect.y = 0
            rect.rotated = False
            
            if (rect.get_width() + min_spacing <= new_bin.width and
                rect.get_length() + min_spacing <= new_bin.length):
                new_bin.rectangles.append(rect)
                bins.append(new_bin)
                placed = True
            else:
                rect.rotated = True
                if (rect.get_width() + min_spacing <= new_bin.width and
                    rect.get_length() + min_spacing <= new_bin.length):
                    new_bin.rectangles.append(rect)
                    bins.append(new_bin)
                    placed = True
        
        if not placed:
            print(f"  WARNING: Could not place {rect.obj_name}")
    
    print(f"  ✓ Created {len(bins)} bins total")
    return bins

print('✓ Packing algorithm updated (stacking by name/dimensions)')

✓ Packing algorithm updated (stacking by name/dimensions)


## 5. Predefined Bin Size

In [7]:
class PredefinedBinPlanner:
    """
    Packing planner with predefined bin sizes.
    Useful when you have specific laydown areas or container types.
    """
    
    def __init__(self, bin_specs, population_size=50, generations=20, mutation_rate=0.1, crossover_rate=0.8):
        """
        Initialize with list of bin specifications.
        
        Args:
            bin_specs: List of dicts with bin info
                Example: [
                    {'id': 'BIN_A', 'width': 200, 'length': 300, 'max_weight': 5000, 'quantity': 2},
                    {'id': 'BIN_B', 'width': 150, 'length': 150, 'max_weight': 3000, 'quantity': 3},
                ]
            population_size: GA population size
            generations: GA generations
            mutation_rate: GA mutation rate
            crossover_rate: GA crossover rate
        """
        self.bin_specs = bin_specs
        self.population_size = population_size
        self.generations = generations
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        self.bin_instances = []
        
        # Create bin instances from specs
        self._create_bin_instances()
    
    def _create_bin_instances(self):
        """Create actual bin instances from specs."""
        self.bin_instances = []
        bin_id = 0
        
        for spec in self.bin_specs:
            for qty in range(spec.get('quantity', 1)):
                bin_obj = Bin(
                    width=spec['width'],
                    length=spec['length'],
                    bin_id=bin_id,
                    min_spacing=0,
                    compatibility_matrix=None
                )
                bin_obj.bin_type = spec['id']
                bin_obj.max_weight = spec.get('max_weight', float('inf'))
                self.bin_instances.append(bin_obj)
                bin_id += 1
    
    def get_bin_specs_table(self):
        """Display bin specifications."""
        print("\n=== PREDEFINED BIN SPECIFICATIONS ===\n")
        print(f"{'Bin Type':<15} {'Width':<10} {'Length':<10} {'Max Weight':<15} {'Quantity':<10}")
        print("-" * 60)
        
        for spec in self.bin_specs:
            width = spec['width']
            length = spec['length']
            max_weight = spec.get('max_weight', 'Unlimited')
            qty = spec.get('quantity', 1)
            print(f"{spec['id']:<15} {width:<10} {length:<10} {str(max_weight):<15} {qty:<10}")
        
        print(f"\nTotal bins available: {sum(s.get('quantity', 1) for s in self.bin_specs)}")
    
    def pack_greedy(self, objects_df, min_spacing=0, compatibility_matrix=None):
        """
        Greedy packing into predefined bins.
        Tries to fit into existing bins before creating new ones.
        """
        print(f"\n{'='*60}")
        print("GREEDY PACKING - Predefined Bins")
        print(f"{'='*60}\n")
        
        # Create rectangles - ONE per object
        rects = []
        for _, row in objects_df.iterrows():
            rect = Rectangle(
                width=float(row['width']),
                length=float(row['length']),
                obj_id=row['object_id'],
                obj_name=row['name'],
                category=row['category'],
                weight=float(row['weight']),
                stackable=row['stackable'] in [True, 'True', 'TRUE'],
                max_stack=int(row['max_stack_height'])
            )
            rects.append(rect)
        
        rects.sort(key=lambda r: r.area(), reverse=True)
        
        # Reset bin instances
        self._create_bin_instances()
        used_bins = []
        failed_objects = []
        
        for rect in rects:
            placed = False
            
            # Try stacking first
            if rect.stackable:
                for bin_obj in used_bins:
                    if placed:
                        break
                    for existing in bin_obj.rectangles:
                        if (existing.obj_name == rect.obj_name and 
                            existing.width == rect.width and
                            existing.length == rect.length and
                            existing.stack_height < existing.max_stack):
                            
                            if compatibility_matrix:
                                if not compatibility_matrix.get(rect.category, {}).get(existing.category, True):
                                    continue
                                
                            rect.x = existing.x
                            rect.y = existing.y
                            rect.rotated = existing.rotated
                            rect.stack_height = existing.stack_height + 1
                            existing.stack_height = rect.stack_height
                            
                            # Check weight limit
                            total_weight = sum(r.weight for r in bin_obj.rectangles) + rect.weight
                            if total_weight <= bin_obj.max_weight:
                                bin_obj.rectangles.append(deepcopy(rect))
                                placed = True
                                break
                            
            # Try existing bins
            if not placed:
                for bin_obj in used_bins:
                    if placed:
                        break
                    
                    if compatibility_matrix:
                        if not bin_obj.can_accept_category(rect.category):
                            continue
                        
                    positions = [(0, 0)]
                    for existing in bin_obj.rectangles:
                        positions.append((existing.x + existing.get_width() + min_spacing, existing.y))
                        positions.append((existing.x, existing.y + existing.get_length() + min_spacing))
                    
                    for x, y in positions:
                        if placed:
                            break
                        
                        # Try normal orientation
                        rect.x = x
                        rect.y = y
                        rect.rotated = False
                        
                        # Check if fits in bin bounds
                        fits_in_bounds = (rect.x + rect.get_width() + min_spacing <= bin_obj.width and
                                         rect.y + rect.get_length() + min_spacing <= bin_obj.length)
                        
                        # Check for overlaps
                        has_overlap = False
                        if fits_in_bounds:
                            for existing in bin_obj.rectangles:
                                if bin_obj._overlaps(rect, existing):
                                    has_overlap = True
                                    break
                                
                        # Check weight
                        total_weight = sum(r.weight for r in bin_obj.rectangles) + rect.weight
                        weight_ok = total_weight <= bin_obj.max_weight
                        
                        if fits_in_bounds and not has_overlap and weight_ok:
                            bin_obj.rectangles.append(deepcopy(rect))
                            placed = True
                            break
                        
                        # Try rotated orientation
                        rect.rotated = True
                        fits_in_bounds = (rect.x + rect.get_width() + min_spacing <= bin_obj.width and
                                         rect.y + rect.get_length() + min_spacing <= bin_obj.length)
                        
                        has_overlap = False
                        if fits_in_bounds:
                            for existing in bin_obj.rectangles:
                                if bin_obj._overlaps(rect, existing):
                                    has_overlap = True
                                    break
                                
                        if fits_in_bounds and not has_overlap and weight_ok:
                            bin_obj.rectangles.append(deepcopy(rect))
                            placed = True
                            break
                        
                        rect.rotated = False
            
            # Use new bin if needed
            if not placed:
                # Find available bin from predefined specs
                available_bins = [b for b in self.bin_instances if b not in used_bins]
                
                if available_bins:
                    new_bin = available_bins[0]
                    rect.x = 0
                    rect.y = 0
                    rect.rotated = False
                    
                    total_weight = rect.weight
                    
                    # Try normal orientation
                    if (rect.get_width() + min_spacing <= new_bin.width and
                        rect.get_length() + min_spacing <= new_bin.length and
                        total_weight <= new_bin.max_weight):
                        new_bin.rectangles.append(deepcopy(rect))
                        used_bins.append(new_bin)
                        placed = True
                    else:
                        # Try rotated orientation
                        rect.rotated = True
                        if (rect.get_width() + min_spacing <= new_bin.width and
                            rect.get_length() + min_spacing <= new_bin.length and
                            total_weight <= new_bin.max_weight):
                            new_bin.rectangles.append(deepcopy(rect))
                            used_bins.append(new_bin)
                            placed = True
            
            # Only add to failed if not placed
            if not placed:
                failed_objects.append(rect)
        
        print(f"✓ Greedy packing complete")
        print(f"  Bins used: {len(used_bins)}")
        print(f"  Objects packed: {sum(len(b.rectangles) for b in used_bins)}")
        print(f"  Objects failed: {len(failed_objects)}")
        
        if failed_objects:
            print(f"\n⚠ Failed to pack {len(failed_objects)} objects:")
            for obj in failed_objects[:10]:
                print(f"    - {obj.obj_name} ({obj.width}x{obj.length})")
            if len(failed_objects) > 10:
                print(f"    ... and {len(failed_objects) - 10} more")
        
        return used_bins
    
    def pack_genetic(self, objects_df, min_spacing=0, compatibility_matrix=None):
        """Genetic Algorithm packing into predefined bins."""
        print(f"\n{'='*60}")
        print("GENETIC ALGORITHM - Predefined Bins")
        print(f"{'='*60}\n")
        
        # Create rectangles
        rects = []
        for _, row in objects_df.iterrows():
            rect = Rectangle(
                width=float(row['width']),
                length=float(row['length']),
                obj_id=row['object_id'],
                obj_name=row['name'],
                category=row['category'],
                weight=float(row['weight']),
                stackable=row['stackable'] in [True, 'True', 'TRUE'],
                max_stack=int(row['max_stack_height'])
            )
            rects.append(rect)
        
        # GA logic here - similar to before but respecting bin specs
        # For now, use greedy as fallback
        print("(GA implementation for predefined bins in progress)")
        return self.pack_greedy(objects_df, min_spacing, compatibility_matrix)

print('✓ PredefinedBinPlanner class created')

✓ PredefinedBinPlanner class created


## 6. Test Predefined Bin Planner

In [ ]:
## 6. Test Predefined Bin Planner

# Define your predefined bin sizes
bin_specifications = [
    {'id': 'LAYDOWN_A', 'width': 400, 'length': 600, 'max_weight': 10000, 'quantity': 2},
    {'id': 'LAYDOWN_B', 'width': 300, 'length': 300, 'max_weight': 5000, 'quantity': 3},
    {'id': 'TRUCK_BED', 'width': 250, 'length': 500, 'max_weight': 8000, 'quantity': 1},
]

# Create the planner
predefined_planner = PredefinedBinPlanner(bin_specifications)

# Display bin specs
predefined_planner.get_bin_specs_table()

# Test with a small subset first
test_objects = objects_df.head(50)  # Start with 50 objects

# Pack using greedy
bins_greedy = predefined_planner.pack_greedy(test_objects, min_spacing=0, compatibility_matrix=compatibility_matrix)

# Display results
print("\n" + "="*60)
print("GREEDY PACKING RESULTS")
print("="*60)
for bin_obj in bins_greedy:
    if bin_obj.rectangles:
        # Calculate UNIQUE footprint area (don't count stacks multiple times)
        unique_positions = set()
        for rect in bin_obj.rectangles:
            unique_positions.add((rect.x, rect.y, rect.width, rect.length, rect.rotated))
        
        unique_area = sum(rect.width * rect.length for rect in bin_obj.rectangles 
                         if (rect.x, rect.y, rect.width, rect.length, rect.rotated) in unique_positions)
        
        # Only count each (x,y) position once
        footprint_area = 0
        seen_positions = set()
        for rect in bin_obj.rectangles:
            pos = (rect.x, rect.y)
            if pos not in seen_positions:
                footprint_area += rect.get_width() * rect.get_length()
                seen_positions.add(pos)
        
        utilization = (footprint_area / (bin_obj.width * bin_obj.length) * 100)
        weight = sum(r.weight for r in bin_obj.rectangles)
        stacks = sum(1 for r in bin_obj.rectangles if r.stack_height > 1)
        
        print(f"\n{bin_obj.bin_type} (Bin {bin_obj.bin_id}):")
        print(f"  Items: {len(bin_obj.rectangles)}")
        print(f"  Stacked items: {stacks}")
        print(f"  Footprint Utilization: {utilization:.1f}%")
        print(f"  Weight: {weight}kg / {bin_obj.max_weight}kg")
        print(f"  Items: {[r.obj_name for r in bin_obj.rectangles[:5]]}{'...' if len(bin_obj.rectangles) > 5 else ''}")

In [ ]:
## 6b. Full Test with Better Diagnostics

# Test with ALL 50 objects
test_objects = objects_df.head(50)

# Pack using greedy
bins_greedy = predefined_planner.pack_greedy(test_objects, min_spacing=0, compatibility_matrix=compatibility_matrix)

# Display detailed results
print("\n" + "="*60)
print("DETAILED GREEDY PACKING RESULTS")
print("="*60)

total_items = 0
total_weight = 0
total_footprint = 0
bin_capacity = 400 * 600

for bin_obj in bins_greedy:
    if bin_obj.rectangles:
        # Calculate footprint (unique x,y positions)
        seen_positions = set()
        footprint_area = 0
        for rect in bin_obj.rectangles:
            pos = (rect.x, rect.y)
            if pos not in seen_positions:
                footprint_area += rect.get_width() * rect.get_length()
                seen_positions.add(pos)
        
        utilization = (footprint_area / (bin_obj.width * bin_obj.length) * 100)
        weight = sum(r.weight for r in bin_obj.rectangles)
        stacks = sum(1 for r in bin_obj.rectangles if r.stack_height > 1)
        
        print(f"\n{bin_obj.bin_type} (Bin {bin_obj.bin_id}):")
        print(f"  Items: {len(bin_obj.rectangles)} ({stacks} stacked)")
        print(f"  Footprint: {utilization:.1f}%")
        print(f"  Weight: {weight}kg / {bin_obj.max_weight}kg ({weight/bin_obj.max_weight*100:.1f}%)")
        
        total_items += len(bin_obj.rectangles)
        total_weight += weight
        total_footprint += footprint_area

print("\n" + "="*60)
print("SUMMARY:")
print("="*60)
print(f"Total items packed: {total_items}")
print(f"Total weight: {total_weight}kg")
print(f"Bins used: {len(bins_greedy)}/{sum(s.get('quantity', 1) for s in bin_specifications)}")
print(f"Average footprint utilization: {total_footprint / (bin_obj.width * bin_obj.length * len(bins_greedy)) * 100:.1f}%")

In [ ]:
## 6c. Full Dataset Test

# Test with ALL 250 objects
print("\nProcessing ALL 250 objects...\n")

# Reset planner for full dataset
predefined_planner = PredefinedBinPlanner(bin_specifications)

# Pack all objects
bins_greedy_full = predefined_planner.pack_greedy(objects_df, min_spacing=0, compatibility_matrix=compatibility_matrix)

# Display detailed results
print("\n" + "="*60)
print("FULL DATASET RESULTS (250 objects)")
print("="*60)

total_items = 0
total_weight = 0
total_footprint = 0

for bin_obj in bins_greedy_full:
    if bin_obj.rectangles:
        # Calculate footprint (unique x,y positions)
        seen_positions = set()
        footprint_area = 0
        for rect in bin_obj.rectangles:
            pos = (rect.x, rect.y)
            if pos not in seen_positions:
                footprint_area += rect.get_width() * rect.get_length()
                seen_positions.add(pos)
        
        utilization = (footprint_area / (bin_obj.width * bin_obj.length) * 100)
        weight = sum(r.weight for r in bin_obj.rectangles)
        stacks = sum(1 for r in bin_obj.rectangles if r.stack_height > 1)
        max_stack_height = max((r.stack_height for r in bin_obj.rectangles), default=1)
        
        print(f"\n{bin_obj.bin_type} (Bin {bin_obj.bin_id}):")
        print(f"  Items: {len(bin_obj.rectangles)} ({stacks} stacked, max height: {max_stack_height})")
        print(f"  Footprint: {utilization:.1f}%")
        print(f"  Weight: {weight}kg / {bin_obj.max_weight}kg ({weight/bin_obj.max_weight*100:.1f}%)")
        
        total_items += len(bin_obj.rectangles)
        total_weight += weight
        total_footprint += footprint_area

print("\n" + "="*60)
print("SUMMARY:")
print("="*60)
print(f"Total items packed: {total_items}/{len(objects_df)}")
print(f"Total weight: {total_weight}kg")
print(f"Bins used: {len(bins_greedy_full)}/{sum(s.get('quantity', 1) for s in bin_specifications)}")
if len(bins_greedy_full) > 0:
    avg_util = total_footprint / (bin_obj.width * bin_obj.length * len(bins_greedy_full)) * 100
    print(f"Average footprint utilization: {avg_util:.1f}%")

## 7 Analyze Required Bin Sizes

In [ ]:
## 7. Analyze Required Bin Sizes

print("\n" + "="*60)
print("BIN SIZE ANALYSIS - What do you actually need?")
print("="*60)

# Total area needed
total_area = objects_df['width'].astype(float) * objects_df['length'].astype(float)
print(f"\nTotal item area: {total_area.sum():.0f} sq units")

# By category
print("\nArea by category:")
for cat in sorted(objects_df['category'].unique()):
    cat_df = objects_df[objects_df['category'] == cat]
    cat_area = (cat_df['width'].astype(float) * cat_df['length'].astype(float)).sum()
    cat_items = len(cat_df)
    print(f"  {cat}: {cat_area:.0f} sq units ({cat_items} items)")

# Largest items
print("\nLargest individual items:")
objects_df['area'] = objects_df['width'].astype(float) * objects_df['length'].astype(float)
largest = objects_df.nlargest(5, 'area')[['name', 'width', 'length', 'area']]
for _, row in largest.iterrows():
    print(f"  {row['name']}: {row['width']:.0f}x{row['length']:.0f} = {row['area']:.0f}")

# Recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS:")
print("="*60)
print("\nOption 1: Increase bin sizes")
print("  - LAYDOWN_A: 400x600 → 500x800 (or larger)")
print("  - LAYDOWN_B: 300x300 → 400x400")
print("  - TRUCK_BED: 250x500 → 300x600")

print("\nOption 2: Add more bins of same size")
print("  - Currently using 6 bins, you have 6 available")
print("  - Consider specifying more bins in bin_specifications")

print("\nOption 3: Create specialized bins for large items")
print("  - Plastic Sheets need at least 95x189 space")
print("  - Steel Beams need at least 32x430 space")

# Calculate minimum bin size for all items
max_width = objects_df['width'].astype(float).max()
max_length = objects_df['length'].astype(float).max()
print(f"\nMinimum single-bin size needed: {max_width:.0f}x{max_length:.0f}")

## 8. Import Bin Sizes from Excel

In [2]:
## 8. Import Bin Specifications from Excel

import openpyxl

def load_bin_specs_from_excel(excel_path, sheet_name='Bins'):
    """
    Load bin specifications from Excel.
    
    Expected columns:
    - id (string): Bin type identifier
    - width (float): Bin width
    - length (float): Bin length
    - max_weight (float): Maximum weight capacity
    - quantity (int): Number of bins of this type
    """
    df = pd.read_excel(excel_path, sheet_name=sheet_name)
    
    bin_specs = []
    for _, row in df.iterrows():
        spec = {
            'id': row['id'],
            'width': float(row['width']),
            'length': float(row['length']),
            'max_weight': float(row['max_weight']),
            'quantity': int(row['quantity'])
        }
        bin_specs.append(spec)
    
    return bin_specs

# Usage:
# bin_specs = load_bin_specs_from_excel('bin_specifications.xlsx')
# planner = PredefinedBinPlanner(bin_specs)

print('✓ Excel import function ready')

✓ Excel import function ready


## 9. Import Bin Sizes from DXF

In [3]:
## 9. Import Bin Specifications from DXF

def load_bin_specs_from_dxf(dxf_path):
    """
    Load bin specifications from DXF file.
    
    Expected DXF structure:
    - Rectangles with specific layer naming convention
    - Attributes stored in block attributes or layer names
    
    Layer naming: BIN_{ID}_{WIDTH}_{LENGTH}_{MAXWEIGHT}_{QTY}
    Example: BIN_LAYDOWN_A_400_600_10000_2
    """
    try:
        dxf = ezdxf.readfile(dxf_path)
        bin_specs = []
        
        # Parse from layer names or block attributes
        for entity in dxf.modelspace().query('LWPOLYLINE'):
            # Example: extract from layer name
            layer = entity.dxf.layer
            if layer.startswith('BIN_'):
                parts = layer.split('_')
                if len(parts) >= 6:
                    spec = {
                        'id': '_'.join(parts[1:-4]),
                        'width': float(parts[-4]),
                        'length': float(parts[-3]),
                        'max_weight': float(parts[-2]),
                        'quantity': int(parts[-1])
                    }
                    bin_specs.append(spec)
        
        return bin_specs
    
    except Exception as e:
        print(f"Error loading DXF: {e}")
        return []

# Usage:
# bin_specs = load_bin_specs_from_dxf('bin_layout.dxf')
# planner = PredefinedBinPlanner(bin_specs)

print('✓ DXF import function ready')

✓ DXF import function ready


In [1]:
import openpyxl